# ITAMAE: usage walkthrough

This notebook exercises the main shared numerical APIs, a toy `PopulationPipeline`, catalogue inspection/export, and spatial helpers. Run **Restart Kernel and Run All**. The grids below are teaching examples, not convergence settings or calibrated SASHIMI physics.

## Setup
From the `itamae-migration` checkout, create a Python 3.12 environment and install the project with its optional backends plus notebook dependencies:
```sh
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install '.[full]' matplotlib h5py numexpr tqdm nbformat nbclient ipykernel
```
Open this notebook with the `python3` kernel from that environment. No notebook cell downloads data.


In [ ]:
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from itamae.provenance import source_revision

display({"itamae": {"version": version("itamae"), "source_revision": source_revision("itamae")}})
print(sys.version)
workspace = tempfile.TemporaryDirectory(prefix="itamae-walkthrough-")
work = Path(workspace.name)

## Units and cosmology
Canonical lengths are Mpc and masses are solar masses at the ITAMAE boundary.


In [ ]:
import astropy.units as u

from itamae.cosmology import NativeFlatLCDM
from itamae.cosmology.colossus import ColossusCosmology
from itamae.units import NativeUnits
from itamae.units.astropy import AstropyUnits

np.testing.assert_allclose(AstropyUnits().to_internal(2000 * u.kpc, "length"), 2.0)
NativeUnits().validate([1e10, 1e12], "mass")
cosmo = NativeFlatLCDM()
z = np.array([0.0, 1.0, 3.0])
np.testing.assert_allclose(cosmo.growth_factor(0.0), 1.0)
display({"z": z, "H": cosmo.H(z), "growth": cosmo.growth_factor(z), "time": cosmo.cosmic_time(z)})
print("optional Colossus H:", ColossusCosmology("planck18").H(z))

## NFW profile and inversion


In [ ]:
from itamae.halo import NFWProfile, invert_nfw_mass_function, nfw_mass_function

profile = NFWProfile(r_s=0.02, rho_s=1e15)
radius = np.geomspace(0.001, 0.2, 30)
if not np.all(np.diff(profile.enclosed_mass(radius)) > 0):
    raise RuntimeError("NFW enclosed mass is not monotonic")
x = np.geomspace(0.01, 100.0, 20)
np.testing.assert_allclose(invert_nfw_mass_function(nfw_mass_function(x)), x, rtol=1e-8)
plt.loglog(radius, profile.density(radius))
plt.xlabel("r [Mpc]")
plt.ylabel("Density [Msun/Mpc^3]")
plt.show()

## Power, variance, and cache round trip


In [ ]:
from itamae.power import SharpKWindow, TabulatedPowerSpectrum
from itamae.variance import (
    IntegratedVarianceModel,
    load_variance_cache,
    save_variance_cache,
    variance_cache_key,
)

k = np.logspace(-3, 3, 6001)
power = TabulatedPowerSpectrum(k, np.ones_like(k), identifier="walkthrough:constant-power")
variance = IntegratedVarianceModel(
    power=power,
    window=SharpKWindow(),
    rho_mean=3 / (4 * np.pi),
    k_min=1e-3,
    k_max=1e3,
    n_k=6001,
    filter_scale=1.0,
    growth_function=lambda z: 1 / (1 + np.asarray(z)),
    growth_identifier="toy-growth",
)
masses = np.geomspace(1e-3, 1e3, 12)
values = variance.variance(masses)
key = variance_cache_key(
    variance.identifier, masses, backend_identifier="numpy", settings={"example": "toy"}
)
save_variance_cache(work / "variance.npz", key=key, mass=masses, variance=values)
loaded_mass, loaded_variance = load_variance_cache(work / "variance.npz", expected_key=key)
np.testing.assert_array_equal(loaded_mass, masses)
np.testing.assert_array_equal(loaded_variance, values)

## Evolution solver


In [ ]:
from itamae.evolution import shanks_transform, solve_evolution

times = np.linspace(0.0, 1.0, 21)
solution = solve_evolution(lambda t, state, rate: -rate * state, [1.0], times, args=(2.0,))[:, 0]
np.testing.assert_allclose(solution, np.exp(-2 * times), rtol=1e-5)
np.testing.assert_allclose(shanks_transform(1.0, 1.5, 1.75), 2.0)
plt.plot(times, solution, "o", label="solver")
plt.plot(times, np.exp(-2 * times), label="analytic")
plt.legend()
plt.show()

## Population pipeline and weighted catalogue
The callbacks below are deliberately toy laws; they demonstrate execution and catalogue contracts only.


In [ ]:
from itamae.execution import PopulationPipeline
from itamae.measure import build_accretion_batch
from itamae.types import CatalogMetadata, WeightedSubhaloCatalog

batch = build_accretion_batch(
    np.array([1.0, 2.0, 3.0, 4.0]),
    0.5,
    np.full(4, 5.0),
    np.array([0.2, 0.3, 0.4, 0.5]),
    np.ones(4),
    mvir_acc=np.array([1.1, 2.2, 3.3, 4.4]),
)
pipeline = PopulationPipeline(
    initialize=lambda batch, context: {"initial": batch.mvir_acc},
    evolve=lambda batch, initial, context: {"m_bound": initial["initial"] * context},
    survival=lambda batch, initial, evolved, context: {"surviving": evolved["m_bound"] > 1},
    columns=lambda batch, initial, evolved, survival, context: {
        "m200_acc": batch.m200_acc,
        "m_bound": evolved["m_bound"],
    },
)
execution = pipeline.execute([batch], contexts=[0.8])
catalog = execution.to_catalog(
    CatalogMetadata(model_identifier="walkthrough:toy", backend_identifier="numpy"),
    view="surviving",
)
np.testing.assert_allclose(catalog.columns["m_bound"], batch.mvir_acc * 0.8)
if np.any(catalog.weight_final < 0):
    raise RuntimeError("Catalogue has negative effective counts")
path = work / "catalog.npz"
catalog.to_npz(path)
restored = WeightedSubhaloCatalog.from_npz(path)
for name in catalog.columns:
    np.testing.assert_array_equal(restored.columns[name], catalog.columns[name])
display(catalog.metadata)

## Spatial/orbit helpers


In [ ]:
from itamae.spatial import orbit_radial_measure, radial_period, turning_points


def potential(r):
    return -1 / np.asarray(r)


energy = -0.5
angular_momentum = np.sqrt(0.75)
rp, ra = turning_points(potential, energy, angular_momentum, 0.1, 3.0)
np.testing.assert_allclose([rp, ra], [0.5, 1.5], rtol=1e-9)
period = radial_period(potential, energy, angular_momentum, rp, ra)
np.testing.assert_allclose(period, 2 * np.pi, rtol=1e-8)
measure = orbit_radial_measure(np.linspace(rp, ra, 17), potential, energy, angular_momentum, rp, ra)
np.testing.assert_allclose(measure.weight.sum(), 1.0)
print("turning points, period:", rp, ra, period)

## Coverage / limitations

| Area | Demonstrated here | Not claimed here |
|---|---|---|
| Units/cosmology | Native + Astropy + Colossus smoke checks | cosmology calibration |
| Halo | NFW profile/inversion | variant-specific structure |
| Power/variance | tabulated sharp-k path + cache | production spectra |
| Evolution | generic ODE + Shanks | SASHIMI tidal calibration |
| Execution/catalogue | callback pipeline, weights, NPZ round trip | a calibrated DM model |
| Spatial | spherical orbit helper | phase-space population model |

Current migration gaps such as W/F `PopulationPipeline` integration and model-composition hardening remain tracked in the family migration epic.
